In [5]:
from pathlib import Path
import sys

# 1. CONNEXION AU DRIVE
if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy/"
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = str(Path.cwd().parent) + "\\"
    print(f"Chemin projet : {PROJECT_DIR}")
sys.path.append(PROJECT_DIR)

import numpy as np
import pandas as pd

# --- 2. TES VARIABLES DE LA NUIT (À changer à 3h du mat) ---
MATCH_ID = 3    # Le numéro du match à pronostiquer
MON_GAP_1 = 0   # Gap avec Bob
MON_GAP_2 = 0   # Gap avec le Peloton
HAS_BOOSTER = 1    # 1 si dispo, 0 sinon

# Variables techniques
GAP_MIN = -600
GAP_MAX = 400
GAP_OFFSET = -GAP_MIN

def get_night_decision():
    print(f"\n" + "="*45)
    print(f"🔮 ORACLE MPP - CONSULTATION NOCTURNE 🔮")
    print("="*45)
    
    # --- 3. LES CHEMINS ABSOLUS SUR LE DRIVE ---
    # (Adapte 'MPPAnalysis' si ton dossier s'appelle autrement sur ton Drive)
    base_dir = PROJECT_DIR + "data/"
    file_path = f"{base_dir}/Abaque_Match_{MATCH_ID}.npz" 
    csv_path = f"{base_dir}/CDM_2026.csv"
    
    # --- 4. SANITY CHECK : AFFICHAGE DU MATCH ---
    try:
        df = pd.read_csv(csv_path)
        # On récupère la ligne correspondant au match (Index = ID - 1)
        match_row = df.iloc[MATCH_ID - 1]
        team_a = str(match_row['team_A']).replace('_', ' ').title()
        team_b = str(match_row['team_B']).replace('_', ' ').title()
        phase = str(match_row['phase'])
        
        print(f"\n⚽ MATCH {MATCH_ID} ({phase}) : {team_a} vs {team_b}")
    except Exception as e:
        print(f"\n⚠️ Impossible de charger le CSV pour lire les équipes : {e}")
        print("Assure-toi que le chemin csv_path est correct.")
        return

    # --- 5. CHARGEMENT DE L'ABAQUE ---
    try:
        data = np.load(file_path)
        Q_table = data['q_table']
        ev_actions = data['ev_actions']
    except FileNotFoundError:
        print(f"\n❌ Erreur : Impossible de trouver l'Abaque pour le Match {MATCH_ID}.")
        print(f"Chemin cherché : {file_path}")
        return

    # Conversion et Clipping des Gaps
    safe_g1 = max(GAP_MIN, min(GAP_MAX, MON_GAP_1))
    safe_g2 = max(GAP_MIN, min(GAP_MAX, MON_GAP_2))
    
    idx_g1 = safe_g1 + GAP_OFFSET
    idx_g2 = safe_g2 + GAP_OFFSET
    
    noms_choix = [f"1 ({team_a})", "N (Nul)", f"2 ({team_b})"]
    
    print(f"\n📊 État Actuel : Gap Bob ({MON_GAP_1}) | Gap Peloton ({MON_GAP_2})")
    if safe_g1 != MON_GAP_1 or safe_g2 != MON_GAP_2:
        print(f"⚠️ Gaps extrêmes détectés : clippés aux limites de la grille ({safe_g1} / {safe_g2})")
    
    print("\n--- ANALYSE DE L'ORACLE ---")
    
    if HAS_BOOSTER:
        wr_keep = Q_table[idx_g1, idx_g2, :, 1]
        wr_use = Q_table[idx_g1, idx_g2, :, 2]
        
        for i in range(3):
            print(f"Choix {noms_choix[i]} :")
            print(f"  -> Garder Booster   = {wr_keep[i]*100:.2f}% de victoire finale")
            print(f"  -> Utiliser Booster = {wr_use[i]*100:.2f}% de victoire finale | EV(x2) = {ev_actions[i]*2:.2f}")
            
        best_keep_idx = np.argmax(wr_keep)
        best_use_idx = np.argmax(wr_use)
        
        if wr_use[best_use_idx] > wr_keep[best_keep_idx]:
            gain_wr = (wr_use[best_use_idx] - wr_keep[best_keep_idx]) * 100
            print(f"\n🚀 RECOMMANDATION : Jouer {noms_choix[best_use_idx]} ET UTILISER LE BOOSTER x2 (Gain stratégique: +{gain_wr:.2f}%)")
        else:
            print(f"\n✅ RECOMMANDATION : Jouer {noms_choix[best_keep_idx]} (Garder le Booster)")
            
    else:
        wr_base = Q_table[idx_g1, idx_g2, :, 0]
        
        for i in range(3):
            print(f"Choix {noms_choix[i]} : Win Rate = {wr_base[i]*100:.2f}% | EV = {ev_actions[i]:.2f}")
            
        best_action = np.argmax(wr_base)
        print(f"\n✅ RECOMMANDATION : Jouer {noms_choix[best_action]}")

# Exécution
get_night_decision()

💻 Environnement Local (VS Code) détecté.
Chemin projet : d:\Documents\Code\MonPetitPronoStrategy\

🔮 ORACLE MPP - CONSULTATION NOCTURNE 🔮

⚽ MATCH 3 (Poule_J1) : Canada vs Bosnie

📊 État Actuel : Gap Bob (0) | Gap Peloton (0)

--- ANALYSE DE L'ORACLE ---
Choix 1 (Canada) :
  -> Garder Booster   = 40.61% de victoire finale
  -> Utiliser Booster = 36.37% de victoire finale | EV(x2) = 68.56
Choix N (Nul) :
  -> Garder Booster   = 40.51% de victoire finale
  -> Utiliser Booster = 36.27% de victoire finale | EV(x2) = 63.14
Choix 2 (Bosnie) :
  -> Garder Booster   = 40.11% de victoire finale
  -> Utiliser Booster = 35.42% de victoire finale | EV(x2) = 50.70

✅ RECOMMANDATION : Jouer 1 (Canada) (Garder le Booster)
